# Conformal Prediction (MC-Dropout-normalized) -- uncapped RUL

Same method as `conformal_prediction_FIXED.ipynb`, trained and calibrated on uncapped RUL -- this is the headline result for the abstention comparison.

In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\processed_train_uncapped.csv")

np.random.seed(42)

engine_ids = df["Engine_ID"].unique()
np.random.shuffle(engine_ids)

train_ids = engine_ids[:70]
val_ids = engine_ids[70:85]
test_ids = engine_ids[85:]

train_df = df[df["Engine_ID"].isin(train_ids)].copy()
val_df = df[df["Engine_ID"].isin(val_ids)].copy()
test_df = df[df["Engine_ID"].isin(test_ids)].copy()

print("Training engines:", train_df["Engine_ID"].nunique())
print("Validation engines:", val_df["Engine_ID"].nunique())
print("Testing engines:", test_df["Engine_ID"].nunique())


Training engines: 70
Validation engines: 15
Testing engines: 15


In [3]:
FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]

TARGET = "RUL"


In [4]:
WINDOW = 30

def create_sequences(data, features, target, window=30):
    X = []
    y = []

    for engine_id, engine_data in data.groupby("Engine_ID"):

        engine_data = engine_data.sort_values("Cycle")

        feature_values = engine_data[features].values.astype("float32")
        target_values = engine_data[target].values.astype("float32")

        for i in range(window, len(engine_data)):
            X.append(feature_values[i-window:i])
            y.append(target_values[i])

    return np.array(X), np.array(y)


In [5]:
X_train, y_train = create_sequences(train_df, FEATURES, TARGET, WINDOW)
X_val, y_val = create_sequences(val_df, FEATURES, TARGET, WINDOW)
X_test, y_test = create_sequences(test_df, FEATURES, TARGET, WINDOW)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)


(12216, 30, 17)
(2720, 30, 17)
(2695, 30, 17)


## Train the point-prediction model

In [6]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Bidirectional,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(42)

bilstm = Sequential([

    Input(shape=(WINDOW, len(FEATURES))),

    Bidirectional(
        LSTM(
            64,
            return_sequences=True
        )
    ),

    Dropout(0.2),

    Bidirectional(
        LSTM(32)
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

bilstm.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

bilstm.summary()


C:\Users\Shivam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 30, 128)        │        41,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,313 (333.25 KB)

 Trainable params: 85,313 (333.25 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

history_bilstm = bilstm.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,
    batch_size=64,

    shuffle=False,

    callbacks=[
        early_stopping,
        reduce_lr
    ],

    verbose=1
)


Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - loss: 6521.6021 - mae: 63.5579 - val_loss: 3668.8582 - val_mae: 49.9265 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 3830.4214 - mae: 50.2484 - val_loss: 3659.0896 - val_mae: 50.2710 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - loss: 3830.8792 - mae: 50.3634 - val_loss: 3659.0508 - val_mae: 50.2736 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 3825.5227 - mae: 50.2897 - val_loss: 3659.1018 - val_mae: 50.2752 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 3724.8284 - mae: 49.2666 - val_loss: 2716.7637 - val_mae: 40.9413 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - loss: 2364.9702 - mae: 37.1516 - val_loss: 1709.7606 - val_mae: 33.3021 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - loss: 1908.0011 - mae: 32.2021 

## Per-point difficulty signal (MC Dropout)

In [8]:
def mc_dropout_predict(model, X, n_passes=100):

    predictions = []

    for _ in range(n_passes):

        pred = model(
            X,
            training=True
        ).numpy().ravel()

        predictions.append(pred)

    predictions = np.array(predictions)

    mean_pred = predictions.mean(axis=0)
    std_pred = predictions.std(axis=0)

    return mean_pred, std_pred


## Calibrate on `val_df`

`ALPHA = 0.10` targets 90% coverage, same as the capped version.

In [9]:
val_mean, val_std = mc_dropout_predict(
    bilstm,
    X_val,
    n_passes=100
)

EPS = 1e-6

nonconformity = np.abs(y_val - val_mean) / (val_std + EPS)

ALPHA = 0.10  # target 90% coverage

q = np.quantile(nonconformity, 1 - ALPHA)

print("Conformal quantile (q):", q)


Conformal quantile (q): 6.120702


In [10]:
cal_mean, cal_std = mc_dropout_predict(
    bilstm,
    X_test,
    n_passes=100
)

cal_interval_width = q * cal_std

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Held-out split:")
print("MAE :", mean_absolute_error(y_test, cal_mean))
print("RMSE:", np.sqrt(mean_squared_error(y_test, cal_mean)))
print("R\u00b2  :", r2_score(y_test, cal_mean))

coverage = (
    np.abs(y_test - cal_mean) <= cal_interval_width
).mean()

print(f"\nEmpirical coverage on held-out split: {coverage:.1%} (target: {1 - ALPHA:.0%})")


Held-out split:
MAE : 27.935453414916992
RMSE: 38.65416734739364
R²  : 0.6349290609359741

Empirical coverage on held-out split: 88.2% (target: 90%)


## Accept/abstain threshold, calibrated on `val_df`

In [11]:
ACCEPT_RATE = 0.80

val_interval_width = q * val_std

cp_threshold = np.quantile(
    val_interval_width,
    ACCEPT_RATE
)

print("Conformal interval-width threshold:", cp_threshold)


Conformal interval-width threshold: 64.39662


## Official test set (uncapped)

In [16]:
test_df_final = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\processed_test.csv")
output_df = pd.read_csv(
    r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\RUL_FD001.csv")
output_df["Engine_ID"] = np.arange(1, len(output_df) + 1)
# NOTE: ground truth left uncapped on purpose -- this notebook trains on
# uncapped RUL, so evaluation has to stay uncapped too, or predictions and
# ground truth would no longer be answering the same question.

def create_test_sequences(test_df, features, window=30):
    X, engine_ids = [], []
    for engine_id, engine_data in test_df.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")
        values = engine_data[features].values.astype("float32")
        if len(values) >= window:
            X.append(values[-window:])
            engine_ids.append(engine_id)
    return np.array(X), np.array(engine_ids)

X_test_final, test_engine_ids = create_test_sequences(test_df_final, FEATURES, WINDOW)


In [17]:
cp_test_mean, cp_test_std = mc_dropout_predict(
    bilstm,
    X_test_final,
    n_passes=100
)

cp_test_interval_width = q * cp_test_std

pred_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Predicted_RUL": cp_test_mean,
    "IntervalWidth": cp_test_interval_width
})

comparison = pred_df.merge(output_df, on="Engine_ID")

actual = comparison["RUL"].values
cp_test_mean = comparison["Predicted_RUL"].values
cp_test_interval_width = comparison["IntervalWidth"].values


In [18]:
def nasa_score(actual, predicted):
    """NASA / PHM08 asymmetric RUL scoring function (Saxena et al., 2008).
    Penalizes late predictions (predicted RUL > actual RUL) more heavily
    than early ones, since a late prediction means maintenance would be
    scheduled after the engine could already have failed.
    """
    d = np.asarray(predicted) - np.asarray(actual)
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(s)


In [19]:
print("100% coverage:")
print("MAE  :", mean_absolute_error(actual, cp_test_mean))
print("RMSE :", np.sqrt(mean_squared_error(actual, cp_test_mean)))
print("R\u00b2   :", r2_score(actual, cp_test_mean))
print("Score:", nasa_score(actual, cp_test_mean))

official_coverage = (
    np.abs(actual - cp_test_mean) <= cp_test_interval_width
).mean()

print(f"\nEmpirical coverage on official test set: {official_coverage:.1%} (target: {1 - ALPHA:.0%})")


100% coverage:
MAE  : 20.023984909057617
RMSE : 27.060215243479448
R²   : 0.5759637355804443
Score: 8630.417085944406

Empirical coverage on official test set: 96.0% (target: 90%)


## Accept / abstain sweep

In [20]:
accept = cp_test_interval_width <= cp_threshold

print("Accepted :", accept.sum())
print("Abstained:", (~accept).sum())
print("Coverage :", accept.mean())


Accepted : 84
Abstained: 16
Coverage : 0.84


In [21]:
y_true_accept = actual[accept]
y_pred_accept = cp_test_mean[accept]

mae = mean_absolute_error(y_true_accept, y_pred_accept)
rmse = np.sqrt(mean_squared_error(y_true_accept, y_pred_accept))
r2 = r2_score(y_true_accept, y_pred_accept)
score = nasa_score(y_true_accept, y_pred_accept)

print("Accepted predictions:")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R\u00b2   :", r2)
print("Score:", score)


Accepted predictions:
MAE  : 17.64359474182129
RMSE : 24.53319309581536
R²   : 0.6261756420135498
Score: 6412.05023336275


In [22]:
for rate in [0.9, 0.8, 0.7]:

    threshold = np.quantile(
        val_interval_width,
        rate
    )

    accept = cp_test_interval_width <= threshold

    rmse = np.sqrt(
        mean_squared_error(
            actual[accept],
            cp_test_mean[accept]
        )
    )

    score = nasa_score(
        actual[accept],
        cp_test_mean[accept]
    )

    print(
        f"Acceptance={rate:.0%} | "
        f"Accepted={accept.sum()} | "
        f"RMSE={rmse:.3f} | "
        f"Score={score:.1f}"
    )


Acceptance=90% | Accepted=93 | RMSE=25.176 | Score=6912.2
Acceptance=80% | Accepted=84 | RMSE=24.533 | Score=6412.1
Acceptance=70% | Accepted=79 | RMSE=21.954 | Score=3934.7


In [23]:
import os

METHOD = "conformal"
SCENARIO = "clean_uncapped"

RESULTS_DIR = r"C:\Users\Shivam\Desktop\Capstone_updated\results"
os.makedirs(RESULTS_DIR, exist_ok=True)

predicted = cp_test_mean
uncertainty = cp_test_interval_width

results_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Actual_RUL": actual,
    "Predicted_RUL": predicted,
    "Uncertainty": uncertainty
})

out_path = os.path.join(
    RESULTS_DIR,
    f"{METHOD}__{SCENARIO}.csv"
)

results_df.to_csv(out_path, index=False)

print(f"Saved {len(results_df)} rows -> {out_path}")

Saved 100 rows -> C:\Users\Shivam\Desktop\Capstone_updated\results\conformal__clean_uncapped.csv
